# A portfolio analyst agent: MongoDB aggregations + Brave Search

An agent over `sample_analytics`, the banking dataset in Atlas's sample data:
500 customers, 1,746 accounts and 88,000 stock trades across 18 tickers. The
collections reference each other by `account_id`, so the tools are
**aggregation pipelines**: `$lookup` joins customers to their accounts and
trades, `$unwind` and `$group` turn trades into net positions per stock. No
embeddings and no search index of any kind.

**Brave Search** adds the one thing the dataset can't contain: the current
share price, which turns a net position into a valuation. As in
[`mongodb_movies_brave.ipynb`](mongodb_movies_brave.ipynb), the database
decides what exists: the agent looks up on the web only tickers that appear in
the trades, and says so when asked about one that doesn't.

The data is synthetic: trades run from 1962 to 2017 at made-up prices, and
some accounts sell shares they never bought. Positions can be negative, and
the agent reports them as they are.

Needs `MONGODB_CONNECTION` (an Atlas cluster with the sample dataset loaded)
and `BRAVE_API_KEY` in `.env`.

In [2]:
import json
import os
import re
import threading
import time

import httpx
from dotenv import load_dotenv
from IPython.display import Markdown, display
from pymongo import MongoClient
from strands import Agent, tool
from strands.models import BedrockModel

load_dotenv()
YOUR_BEDROCK_MODEL_ID = "us.anthropic.claude-sonnet-5-5"

mongo_client = MongoClient(os.environ["MONGODB_CONNECTION"])
db = mongo_client["sample_analytics"]
customers, accounts, transactions = db["customers"], db["accounts"], db["transactions"]
for coll in (customers, accounts, transactions):
    print(f"{coll.name}: {coll.estimated_document_count():,} documents")

brave_api_key = os.environ.get("BRAVE_API_KEY", "")
print(f"Brave API key available: {'Yes' if brave_api_key else 'No'}")

customers: 500 documents
accounts: 1,746 documents
transactions: 1,746 documents
Brave API key available: Yes


## 1. The data

Three collections linked by `account_id`:

- `customers`: `name`, `email`, `active`, `accounts` (a list of ids) and
  `tier_and_details`, a map of membership tiers with their benefits.
- `accounts`: `account_id`, `limit` and the `products` it holds.
- `transactions`: one document per account with its trades in an array:
  `date`, `symbol`, `transaction_code` (`buy`/`sell`), `amount` (shares) and
  `price`/`total`, which are stored as **strings** and need `$toDouble` before
  any arithmetic.

In [3]:
customer = customers.find_one({"name": "Elizabeth Ray"}, {"_id": 0})
print(json.dumps(customer, indent=1, default=str)[:900], "...")

account = accounts.find_one({"account_id": customer["accounts"][0]}, {"_id": 0})
print(json.dumps(account, indent=1))

trades = transactions.find_one({"account_id": customer["accounts"][0]}, {"_id": 0, "transactions": {"$slice": 2}})
print(json.dumps(trades, indent=1, default=str))

{
 "username": "fmiller",
 "name": "Elizabeth Ray",
 "address": "9286 Bethany Glens\nVasqueztown, CO 22939",
 "birthdate": "1977-03-02 02:20:31",
 "email": "arroyocolton@gmail.com",
 "active": true,
 "accounts": [
  371138,
  324287,
  276528,
  332179,
  422649,
  387979
 ],
 "tier_and_details": {
  "0df078f33aa74a2e9696e0520c1a828a": {
   "tier": "Bronze",
   "id": "0df078f33aa74a2e9696e0520c1a828a",
   "active": true,
   "benefits": [
    "sports tickets"
   ]
  },
  "699456451cc24f028d2aa99d7534c219": {
   "tier": "Bronze",
   "benefits": [
    "24 hour dedicated line",
    "concierge services"
   ],
   "active": true,
   "id": "699456451cc24f028d2aa99d7534c219"
  }
 }
} ...
{
 "account_id": 371138,
 "limit": 9000,
 "products": [
  "Derivatives",
  "InvestmentStock"
 ]
}
{
 "account_id": 371138,
 "transaction_count": 14,
 "bucket_start_date": "1977-12-10 00:00:00",
 "bucket_end_date": "2016-12-04 00:00:00",
 "transactions": [
  {
   "date": "2013-10-18 00:00:00",
   "amount": 3887,

The sample data has no index on `account_id`, so every `$lookup` scans the
joined collection. Small here, but it's the index a real deployment would
have; `create_index` is a no-op when it already exists.

In [4]:
for coll in (accounts, transactions):
    coll.create_index("account_id", name="account_id_1")
print("Indexes on transactions:", [i["name"] for i in transactions.list_indexes()])

Indexes on transactions: ['_id_', 'account_id_1']


## 2. The tools

Three run aggregations and one reads the web:

- `find_customers` filters by name, tier or product and joins the accounts.
- `customer_portfolio` joins a customer's trades and nets buys against sells
  per ticker. Its output is what the web search prices.
- `symbol_activity` is the bank-wide view per ticker, and the gate for the
  web: a ticker it doesn't return isn't in the dataset.
- `web_search` calls the Brave Search API directly with `httpx`.

In [5]:
# tier_and_details is a map keyed by an opaque id; this turns it into a plain
# list of tier names so it can be matched and displayed.
TIERS = {"$map": {"input": {"$objectToArray": "$tier_and_details"}, "as": "t", "in": "$$t.v.tier"}}

# One trade per document after $unwind; these sum shares and cash by side.
IS_BUY = {"$eq": ["$tx.transactions.transaction_code", "buy"]}
IS_SELL = {"$eq": ["$tx.transactions.transaction_code", "sell"]}
TOTAL = {"$toDouble": "$tx.transactions.total"}  # stored as a string

JOIN_ACCOUNTS = {"$lookup": {"from": "accounts", "localField": "accounts", "foreignField": "account_id", "as": "accs"}}
JOIN_TRADES = {"$lookup": {"from": "transactions", "localField": "accounts", "foreignField": "account_id", "as": "tx"}}


def money(value):
    return f"${value:,.2f}"


@tool
def find_customers(
    name: str | None = None,
    tier: str | None = None,
    product: str | None = None,
    limit: int = 10,
) -> str:
    """Find customers by name, membership tier and/or account product, with their accounts.

    Args:
        name: Part of the customer's name, case-insensitive, e.g. "Perkins".
        tier: Membership tier: Bronze, Silver, Gold or Platinum.
        product: An account product: InvestmentStock, Brokerage, CurrencyService,
            InvestmentFund, Derivatives or Commodity.
        limit: Maximum number of customers to return.
    """
    pipeline = [{"$addFields": {"tiers": TIERS}}]
    match = {}
    if name:
        match["name"] = re.compile(re.escape(name), re.IGNORECASE)
    if tier:
        match["tiers"] = re.compile(f"^{re.escape(tier)}$", re.IGNORECASE)
    if match:
        pipeline.append({"$match": match})
    pipeline.append(JOIN_ACCOUNTS)
    if product:
        pipeline.append({"$match": {"accs.products": re.compile(f"^{re.escape(product)}$", re.IGNORECASE)}})
    pipeline += [{"$sort": {"name": 1}}, {"$facet": {"total": [{"$count": "n"}], "rows": [{"$limit": limit}]}}]

    result = next(customers.aggregate(pipeline))
    total = result["total"][0]["n"] if result["total"] else 0
    rows = result["rows"]
    print(f"[find_customers] {json.dumps({k: v for k, v in dict(name=name, tier=tier, product=product).items() if v})}: {total} matches")

    if not rows:
        return "No customers match these filters."

    lines = [f"{total} customers match, showing {len(rows)}:"]
    for c in rows:
        benefits = sorted({b for t in c["tier_and_details"].values() for b in t["benefits"]})
        lines.append(
            f"\n{c['name']} ({c['email']}, {'active' if c.get('active') else 'inactive'}) "
            f"— tiers: {', '.join(sorted(set(c['tiers']))) or 'none'}; benefits: {', '.join(benefits) or 'none'}"
        )
        for a in c["accs"]:
            lines.append(f"  account {a['account_id']}: limit {money(a['limit'])}; products: {', '.join(a['products'])}")
    return "\n".join(lines)


@tool
def customer_portfolio(name: str) -> str:
    """Net stock position per ticker for one customer, across all their accounts.

    Nets every buy against every sell. The data is synthetic, so a position can
    be negative (more shares sold than bought); report it as it is.

    Args:
        name: The customer's full name, case-insensitive, e.g. "Bryce Bowen".
    """
    pipeline = [
        {"$match": {"name": re.compile(f"^{re.escape(name)}$", re.IGNORECASE)}},
        JOIN_TRADES,
        {"$unwind": "$tx"},
        {"$unwind": "$tx.transactions"},
        {"$group": {
            "_id": "$tx.transactions.symbol",
            "bought": {"$sum": {"$cond": [IS_BUY, "$tx.transactions.amount", 0]}},
            "sold": {"$sum": {"$cond": [IS_SELL, "$tx.transactions.amount", 0]}},
            "spent": {"$sum": {"$cond": [IS_BUY, TOTAL, 0]}},
            "received": {"$sum": {"$cond": [IS_SELL, TOTAL, 0]}},
            "trades": {"$sum": 1},
            "first": {"$min": "$tx.transactions.date"},
            "last": {"$max": "$tx.transactions.date"},
        }},
        {"$sort": {"_id": 1}},
    ]
    rows = list(customers.aggregate(pipeline))
    print(f"[customer_portfolio] {name!r}: {len(rows)} tickers")

    if not rows:
        if customers.count_documents({"name": re.compile(f"^{re.escape(name)}$", re.IGNORECASE)}) == 0:
            return f"NOT IN DATASET: no customer named {name!r}."
        return f"{name} has no trades."

    lines = [f"Net positions of {name} (shares bought - sold; cash is spent on buys minus received from sells):"]
    for r in rows:
        lines.append(
            f"- {r['_id'].upper()}: net {r['bought'] - r['sold']:,} shares "
            f"(bought {r['bought']:,}, sold {r['sold']:,}) in {r['trades']} trades, "
            f"{r['first']:%Y-%m-%d} to {r['last']:%Y-%m-%d}; net cash {money(r['received'] - r['spent'])}"
        )
    return "\n".join(lines)


@tool
def symbol_activity(symbol: str | None = None, limit: int = 18) -> str:
    """Bank-wide trading activity per ticker: trades, accounts, traded volume, last trade.
    Call it before any web search about a ticker: one it doesn't list is not in the dataset.

    Args:
        symbol: One ticker, e.g. "AAPL", for a single row. Omit for every ticker.
        limit: Maximum number of tickers to return, largest volume first.
    """
    pipeline = [{"$unwind": "$transactions"}]
    if symbol:
        pipeline.append({"$match": {"transactions.symbol": symbol.lower()}})
    pipeline += [
        {"$group": {
            "_id": "$transactions.symbol",
            "trades": {"$sum": 1},
            "accounts": {"$addToSet": "$account_id"},
            "volume": {"$sum": {"$toDouble": "$transactions.total"}},
            # $top picks the latest trade per group without sorting the 88,000
            # unwound trades first, which exceeds the 32 MB limit of a $sort.
            "last": {"$top": {
                "sortBy": {"transactions.date": -1},
                "output": {"date": "$transactions.date", "price": {"$toDouble": "$transactions.price"}},
            }},
        }},
        {"$project": {"trades": 1, "volume": 1, "last": 1, "accounts": {"$size": "$accounts"}}},
        {"$sort": {"volume": -1}},
        {"$limit": limit},
    ]
    rows = list(transactions.aggregate(pipeline))
    print(f"[symbol_activity] {symbol or 'all'}: {len(rows)} tickers")

    if not rows:
        return f"NOT IN DATASET: no trades for ticker {symbol!r}."
    return "\n".join(
        f"- {r['_id'].upper()}: {r['trades']:,} trades by {r['accounts']} accounts, "
        f"volume {money(r['volume'])}, last trade {r['last']['date']:%Y-%m-%d} at {money(r['last']['price'])}"
        for r in rows
    )


BRAVE_SEARCH_URL = "https://api.search.brave.com/res/v1/web/search"

# Brave's free plan allows one request per second, and Strands runs the tools of
# a turn concurrently: the lock spaces the requests out instead of getting 429s.
BRAVE_MIN_INTERVAL = 1.1
brave_lock = threading.Lock()
brave_last_request = 0.0

# Brave highlights the matched words in the snippets with <strong>.
HTML_TAG = re.compile(r"<[^>]+>")


@tool
def web_search(query: str, count: int = 5) -> str:
    """Search the web with Brave Search for current information on a ticker that IS in
    the dataset: today's share price, the company behind it, recent news.

    Never use it for a ticker that symbol_activity did not list.

    Args:
        query: The search query, e.g. "CSCO stock price today".
        count: Number of results to return (at most 20).
    """
    global brave_last_request

    with brave_lock:
        wait = BRAVE_MIN_INTERVAL - (time.monotonic() - brave_last_request)
        if wait > 0:
            time.sleep(wait)
        response = httpx.get(
            BRAVE_SEARCH_URL,
            params={"q": query, "count": min(count, 20)},
            headers={
                "Accept": "application/json",
                "X-Subscription-Token": brave_api_key,
            },
            timeout=15,
        )
        brave_last_request = time.monotonic()

    response.raise_for_status()
    results = response.json().get("web", {}).get("results", [])
    print(f"[web_search] {query!r}: {len(results)} results")

    if not results:
        return "No web results found."

    return "\n\n".join(
        f"{result['title']}\n{result['url']}\n"
        f"{HTML_TAG.sub('', result.get('description', ''))}"
        for result in results
    )

Tools decorated with `@tool` can still be called as plain functions:

In [6]:
print(customer_portfolio("Bryce Bowen"))

[customer_portfolio] 'Bryce Bowen': 3 tickers
Net positions of Bryce Bowen (shares bought - sold; cash is spent on buys minus received from sells):
- CSCO: net -13,991 shares (bought 71,938, sold 85,929) in 36 trades, 1992-06-15 to 2016-05-24; net cash $562,459.17
- EBAY: net -60,109 shares (bought 52,335, sold 112,444) in 26 trades, 2002-03-07 to 2016-10-28; net cash $915,220.90
- INTC: net -5,114 shares (bought 81,947, sold 87,061) in 36 trades, 1975-01-22 to 2016-05-25; net cash $303,341.01


In [7]:
print(symbol_activity(limit=5))

[symbol_activity] all: 5 tickers
- GOOG: 5,500 trades by 562 accounts, volume $18,110,672,711.55, last trade 2017-01-09 at $807.68
- AMZN: 5,088 trades by 521 accounts, volume $5,193,716,657.13, last trade 2017-01-06 at $788.48
- FB: 4,692 trades by 515 accounts, volume $1,927,212,916.86, last trade 2017-01-09 at $124.21
- IBM: 4,908 trades by 525 accounts, volume $1,573,161,353.99, last trade 2017-01-09 at $161.46
- SAP: 4,787 trades by 495 accounts, volume $1,456,411,290.02, last trade 2017-01-09 at $88.45


## 3. The agent

In [8]:
SYSTEM_PROMPT = """You are an analyst for a bank's customer database in MongoDB (customers, accounts and stock trades up to 2017), with web search for current share prices and company news.

## Rules

1. The dataset decides which customers and tickers exist. Before searching the web about a ticker, confirm it with `symbol_activity` or a customer's portfolio. If a ticker is not in the dataset, say so and do not search the web for it.
2. Use `web_search` only for current information on tickers in the dataset: today's share price, the company behind the ticker, recent news. One search per ticker, with the ticker in the query.
3. To value a position, multiply the net shares from `customer_portfolio` by the current price from the web. Show the price, its source URL and its date; prices change, so say when it was quoted.
4. The trades are synthetic and end in 2017: positions can be negative and historical prices are not real. Report what the data says, and say when a ticker has since changed (renamed, acquired, delisted).
5. Say what comes from the database and what comes from the web. Answer in the language of the question. Keep answers concise.
"""

model = BedrockModel(
    model_id=YOUR_BEDROCK_MODEL_ID,
    region_name=os.getenv("AWS_REGION", "us-west-2"),
    max_tokens=4096,
)


def ask(question):
    # A fresh agent per question, so earlier answers don't leak into the next
    agent = Agent(
        name="Portfolio Analyst",
        model=model,
        system_prompt=SYSTEM_PROMPT,
        tools=[find_customers, customer_portfolio, symbol_activity, web_search],
        callback_handler=None,  # don't also stream the answer to stdout
    )

    response = agent(question)

    # Tools the agent actually called (absent from the dict means never called)
    used = response.metrics.tool_metrics
    usage = response.metrics.accumulated_usage
    display(Markdown(
        "**Tools used:** "
        + (", ".join(f"`{name}` x{m.call_count}" for name, m in used.items()) or "_none_")
        + f" — {usage['inputTokens']:,} input / {usage['outputTokens']:,} output tokens"
    ))

    display(Markdown(str(response)))

Database only, with a `$lookup` from the customer to the accounts:

In [9]:
ask("¿Qué cuentas, productos y beneficios tiene Elizabeth Ray?")

[find_customers] {"name": "Elizabeth Ray"}: 1 matches


**Tools used:** `find_customers` x1 — 3,403 input / 522 output tokens

**Elizabeth Ray** (arroyocolton@gmail.com) es una cliente activa con nivel **Bronze**. Todos estos datos vienen de la base de datos.

**Beneficios:** línea dedicada 24 horas, servicios de concierge y entradas para eventos deportivos.

**Cuentas**

| Cuenta | Límite | Productos |
|---|---|---|
| 276528 | $10,000 | InvestmentFund, InvestmentStock |
| 324287 | $10,000 | Commodity, CurrencyService, Derivatives, InvestmentStock |
| 332179 | $10,000 | Commodity, CurrencyService, InvestmentFund, Brokerage, InvestmentStock |
| 371138 | $9,000 | Derivatives, InvestmentStock |
| 387979 | $10,000 | Brokerage, Derivatives, InvestmentFund, Commodity, InvestmentStock |
| 422649 | $10,000 | CurrencyService, InvestmentStock |

**Productos en total:** tiene los seis productos del banco: InvestmentStock, InvestmentFund, Brokerage, CurrencyService, Derivatives y Commodity. InvestmentStock está en las seis cuentas.

Puedo revisar también su portafolio de acciones y valorarlo con precios actuales, si le sirve.


In [10]:
ask("How many Platinum customers hold a Derivatives product? Name five of them.")

[find_customers] {"tier": "Platinum", "product": "Derivatives"}: 76 matches


**Tools used:** `find_customers` x1 — 19,941 input / 282 output tokens

**76** Platinum customers hold a Derivatives product. This comes from the database, which returned 76 matches with the limit set high enough to list them all.

Five of them:
1. Amanda Rangel
2. Charles Sanchez
3. Curtis Walter
4. Elizabeth Bowers
5. Jacqueline Davis

I counted a customer if the Platinum tier applies to them and at least one of their accounts has Derivatives. Many of them also hold other tiers, such as Gold or Silver. I did not check which tier goes with which account.


Net positions from MongoDB, priced with Brave (one search per ticker):

In [11]:
ask("Show Bryce Bowen's net position per stock and estimate what it is worth at today's prices.")

[customer_portfolio] 'Bryce Bowen': 3 tickers
[symbol_activity] EBAY: 1 tickers
[symbol_activity] CSCO: 1 tickers
[symbol_activity] INTC: 1 tickers
[web_search] 'EBAY stock price today': 3 results
[web_search] 'CSCO stock price today': 3 results
[web_search] 'INTC stock price today': 3 results


**Tools used:** `customer_portfolio` x1, `symbol_activity` x3, `web_search` x3 — 9,331 input / 1,632 output tokens

Bryce Bowen is net short in all three stocks. Marking those shorts at current prices gives a liability of about $9.1 million.

**Net positions (database, trades 1975–2016)**

| Ticker | Bought | Sold | Net shares |
|---|---|---|---|
| CSCO | 71,938 | 85,929 | -13,991 |
| EBAY | 52,335 | 112,444 | -60,109 |
| INTC | 81,947 | 87,061 | -5,114 |

The data is synthetic, so negative positions are reported as they are. All three tickers are in the dataset. Their last recorded trades were in January 2017.

**Current prices (web) and value**

| Ticker | Price | Source | Value |
|---|---|---|---|
| CSCO | $113.30 | [Robinhood](https://robinhood.com/us/en/stocks/CSCO/) (no quote date shown) | -$1,585,180 |
| EBAY | about $117.20, the midpoint of the $115.80–$118.60 range on 2026-10-14 | [Robinhood](https://robinhood.com/us/en/stocks/EBAY/) | -$7,044,775 (range -$6.96M to -$7.13M) |
| INTC | $92.39 as of 2026-10-15 | [Robinhood](https://robinhood.com/us/en/stocks/INTC/) | -$472,482 |
| **Total** | | | **about -$9.10 million** |

**Caveats**
- The search results gave no exact EBAY quote, so I used the midpoint of the day's range. The CSCO quote carried no date.
- The prices are live quotes from the search results and will have moved since.
- The database's own 2017 prices were about $29–36 per share. They are synthetic, so I didn't use them.
- This is a mark-to-market estimate of the net share counts only. It doesn't include the cash from the trades or any cost basis.


A bank-wide aggregation, then the web for the one company it points to:

In [12]:
ask("¿Qué acción mueve más volumen en todo el banco y qué ha pasado con esa empresa últimamente?")

[symbol_activity] all: 1 tickers
[web_search] 'GOOG stock price today Alphabet news': 5 results


**Tools used:** `symbol_activity` x1, `web_search` x1 — 5,575 input / 791 output tokens

**Acción con más volumen (base de datos):** GOOG. Tiene 5.500 operaciones en 562 cuentas, con un volumen de unos 18.110 millones de USD. La última operación fue el 2017-01-09 a 807,68 USD.

**Situación actual (web, resultados de búsqueda):**
- GOOG es la acción Clase C de Alphabet Inc., la matriz de Google.
- Robinhood la cotiza a unos **340,50 USD** y da una capitalización de unos 4,23 billones de USD. Su fecha de referencia es 2026-10-15 (https://robinhood.com/us/en/stocks/GOOG/).
- Yahoo Finance cita una caída del 1,68% en la sesión y señala la presentación de resultados del 28 de octubre como próximo catalizador. Lo atribuye a un tono cauteloso hacia las empresas muy ligadas a la IA (https://finance.yahoo.com/quote/GOOG/).
- CNN menciona un descenso del 1,10% (3,86 USD) frente al cierre anterior. Dice que la acción está en la mitad de su rango de 52 semanas y por encima de su media móvil de 200 días (https://www.cnn.com/markets/stocks/GOOG).
- Las noticias recientes de Yahoo para GOOGL se centran en la IA: titulares de seguridad, restricciones de suministro en infraestructura de IA y objetivos de precio cercanos a 353,8 USD.

**Advertencias:**
- Los precios de los resultados son fragmentos de búsqueda con fechas y horas de cotización distintas. Pueden no ser exactamente simultáneos.
- Los precios de las operaciones en la base de datos son sintéticos, de 2017, y no son datos reales de mercado. No sirven para comparar con el precio actual.
- No he calculado el valor de ninguna posición. Si quieres valorar la cartera de un cliente en GOOG, dime el nombre.


A ticker the bank never traded. The agent checks the dataset and stops there,
without searching the web:

In [13]:
ask("What is Tesla's share price today?")

[symbol_activity] TSLA: 0 tickers


**Tools used:** `symbol_activity` x1 — 3,127 input / 185 output tokens

TSLA (Tesla) isn't in the bank's dataset. The database has no trades for that ticker, so I didn't search the web for its price. I can look up tickers that do appear in the data, or check a customer's portfolio, if you'd like.


## Clean up

The notebook only reads the sample collections; the `account_id` indexes it
created are harmless to leave in place.

In [13]:
mongo_client.close()